# NOTEBOOK 04: Baseline Multiclass Model

## Random Forest for Water Quality Classification

Loads unscaled train/val/test from Notebook 03. Uses Stratified Shuffle Split for cross-validation. Trains Random Forest with RandomizedSearchCV, evaluates on validation and test, saves best model.

In [ ]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
CONFIG = {
    'data_dir': '../processed_data',
    'models_dir': '../models',
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
    'n_iter': 64,
    'rf_param_grid': {
        'n_estimators': [400, 600, 800, 1000, 1200],
        'max_depth': [16, 20, 24, 28, 32, None],
        'min_samples_split': [2, 4, 6],
        'min_samples_leaf': [1, 2, 3],
        'max_features': ['sqrt', 'log2', 0.4, 0.6],
        'class_weight': ['balanced', 'balanced_subsample'],
        'bootstrap': [True],
    },
    'class_names': ['Poor', 'Moderate', 'Good'],
}

cv_split = StratifiedShuffleSplit(n_splits=4, test_size=0.2, random_state=CONFIG['random_state'])

In [ ]:
os.makedirs(CONFIG['models_dir'], exist_ok=True)
base = CONFIG['data_dir']
X_train = pd.read_csv(f"{base}/X_train.csv")
X_val = pd.read_csv(f"{base}/X_val.csv")
X_test = pd.read_csv(f"{base}/X_test.csv")
y_train = pd.read_csv(f"{base}/y_train.csv")['label'].values
y_val = pd.read_csv(f"{base}/y_val.csv")['label'].values
y_test = pd.read_csv(f"{base}/y_test.csv")['label'].values
print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}, Features: {X_train.shape[1]}")

Train: 34360, Val: 11454, Test: 11454, Features: 34


In [ ]:
rf_base = RandomForestClassifier(random_state=CONFIG['random_state'], n_jobs=CONFIG['n_jobs'], class_weight='balanced')
rf_search = RandomizedSearchCV(
    rf_base,
    CONFIG['rf_param_grid'],
    n_iter=CONFIG['n_iter'],
    cv=cv_split,
    scoring='f1_macro',
    random_state=CONFIG['random_state'],
    n_jobs=CONFIG['n_jobs'],
    verbose=1
)
rf_search.fit(X_train, y_train)
rf_model = rf_search.best_estimator_
print(f"RF best params: {rf_search.best_params_}")
print(f"RF best CV F1 macro: {rf_search.best_score_:.4f}")

Fitting 4 folds for each of 64 candidates, totalling 256 fits
RF best params: {'n_estimators': 1200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 20, 'class_weight': 'balanced', 'bootstrap': True}
RF best CV F1 macro: 0.6243


In [17]:
def eval_model(model, X, y, name):
    pred = model.predict(X)
    acc = accuracy_score(y, pred)
    f1 = f1_score(y, pred, average='macro')
    print(f"{name} — Accuracy: {acc:.4f}, Macro F1: {f1:.4f}")
    print(classification_report(y, pred, target_names=CONFIG['class_names']))
    print(confusion_matrix(y, pred))
    return {'accuracy': acc, 'f1_macro': f1, 'predictions': pred}


In [18]:
print("Test set evaluation:")
eval_model(rf_model, X_test, y_test, "RF")

Test set evaluation:
RF — Accuracy: 0.6352, Macro F1: 0.6355
              precision    recall  f1-score   support

        Poor       0.69      0.71      0.70      3780
    Moderate       0.53      0.52      0.52      3894
        Good       0.68      0.68      0.68      3780

    accuracy                           0.64     11454
   macro avg       0.63      0.64      0.64     11454
weighted avg       0.63      0.64      0.63     11454

[[2692  846  242]
 [ 936 2010  948]
 [ 283  923 2574]]


{'accuracy': 0.6352365985681858,
 'f1_macro': 0.6354502133076495,
 'predictions': array([0, 0, 1, ..., 1, 0, 0])}

In [19]:
print("Validation:")
rf_val = eval_model(rf_model, X_val, y_val, "RF")


Validation:
RF — Accuracy: 0.6310, Macro F1: 0.6305
              precision    recall  f1-score   support

        Poor       0.68      0.71      0.69      3780
    Moderate       0.53      0.50      0.51      3894
        Good       0.68      0.69      0.69      3780

    accuracy                           0.63     11454
   macro avg       0.63      0.63      0.63     11454
weighted avg       0.63      0.63      0.63     11454

[[2670  853  257]
 [ 968 1935  991]
 [ 266  892 2622]]


In [ ]:
joblib.dump(rf_model, f"{CONFIG['models_dir']}/best_baseline_model.joblib")
joblib.dump(rf_model, f"{CONFIG['models_dir']}/baseline_rf.joblib")
print("Saved best_baseline_model.joblib and baseline_rf.joblib")

Saved best_baseline_model.joblib and baseline_rf.joblib
